## PageRank y HITS: Funciones

Constantes y funciones compartidas

### Imports

In [ ]:
import json
import numpy as np
import scipy.sparse as sp
from scipy.sparse.csgraph import connected_components
from scipy.sparse.linalg import svds
from scipy.stats import spearmanr

### Constantes

- `datos_dir`: datos que pasan de un notebook al siguiente

- `clus_dir`: salidas de `02_Clustering` (tipologías y etiquetas de KMeans)

- `GF_PKG`: GraphFrames para Spark 3.5 / Scala 2.12, la versión de la imagen del cluster

In [ ]:
SEED = 42
datos_dir = Path.cwd() / "temp" / "datos"
datos_dir.mkdir(parents=True, exist_ok=True)
clus_dir = proyecto / "02_Clustering" / "temp"
GF_PKG = "graphframes:graphframes:0.8.3-spark3.5-s_2.12"
GF_REPO = "https://repos.spark-packages.org"

### Parámetros elegidos

| Parámetro | Valor | Por qué |
|---|---|---|
| Grafo (PR) | bipartito **no dirigido** | En el dirigido E→P todos los proveedores son colgantes y las entidades quedan empatadas |
| `PESO` (PR y HITS) | `n_adj` | Mide recurrencia de la relación sin copiar el ranking de monto |
| `DAMPING` (PR) | 0.85 | Valor clásico; el ranking es estable alrededor de él |
| `TOL` (PR y HITS, local) | 1e-10 (L1) | Local: iterar es barato (272k entradas no nulas) |
| `TOL_SPARK` (PR en Spark) | 1e-5 (L1) | Cada iteración en Spark cuesta 2 jobs; basta para validar contra local (diferencia < 1e-5) |
| `MAX_ITER` (PR y HITS) | 5,000 | Alcanza para d = 0.99 (≈ 2,300 iteraciones) |
| `TOP_K` (comp de ranks) | 20 | Tamaño de los top para comparar rankings (Jaccard) |
| `P_ALTO` (HITS vs PR y análisis) | 0.99 | Percentil desde el que un actor es "muy central" |
| `UMBRAL_HHI` (cautividad) | 0.25 | El mismo de `02_Clustering` para llamar cautiva a una entidad |
| `MIN_PROC_ENT` (cautividad) | 5 | El mismo filtro de entidades de `02_Clustering` |

In [ ]:
PESO = "n_adj"
DAMPING = 0.85
TOL = 1e-10
TOL_SPARK = 1e-5
MAX_ITER = 5000
TOP_K = 20
P_ALTO = 0.99
UMBRAL_HHI = 0.25
MIN_PROC_ENT = 5

### Configuración entre notebooks

`config.json` guarda lo que cada notebook decide (tamaños, peso, damping…)

In [ ]:
def leer_config():
    ruta = datos_dir / "config.json"
    return json.loads(ruta.read_text()) if ruta.exists() else {}

def guardar_config(config):
    (datos_dir / "config.json").write_text(json.dumps(config, indent=1, ensure_ascii=False))

### Spark con GraphFrames

- Se pasa `--packages` por `PYSPARK_SUBMIT_ARGS`, igual que `pyspark --packages`, sin tocar `crear_spark`

- Si no hay red para bajar el paquete, se sigue sin GraphFrames: la implementación propia no depende de él

In [ ]:
def crear_spark_grafos(nombre):
    os.environ["PYSPARK_SUBMIT_ARGS"] = f"--packages {GF_PKG} --repositories {GF_REPO} pyspark-shell"
    try:
        s = crear_spark(nombre)
    except Exception as e:
        print(f"Sin GraphFrames ({type(e).__name__}): se continúa con la implementación propia")
        os.environ.pop("PYSPARK_SUBMIT_ARGS", None)
        return crear_spark(nombre), False
    try:
        from graphframes import GraphFrame
    except ImportError:
        for jar in s.sparkContext.getConf().get("spark.jars", "").split(","):
            if "graphframes" in jar:
                s.sparkContext.addPyFile(jar)
        try:
            from graphframes import GraphFrame
        except ImportError:
            return s, False
    return s, True

### Grafo

- `cargar_bipartito`: vértices ordenados (entidades primero, luego proveedores) con índice entero `idx`

- `pesos`: `binario`, `n_adj`, `monto` (piso de S/1) o `log_monto` (1 + log1p, siempre positivo)

- `matriz_bip`: A (entidades × proveedores);

- `matriz_grafo`: adyacencia N × N dirigida o no

In [ ]:
def cargar_bipartito():
    v = pd.read_parquet(artef_dir / "grafo_bipartito" / "vertices")
    a = pd.read_parquet(artef_dir / "grafo_bipartito" / "aristas")
    v = pd.concat([v[v.tipo == "entidad"].sort_values("id"), v[v.tipo == "proveedor"].sort_values("id")], ignore_index=True)
    v["idx"] = np.arange(len(v))
    n_e = int((v.tipo == "entidad").sum())
    pos = pd.Series(v.idx.to_numpy(), index=v.id)
    a = a.assign(i=a.src.map(pos), j=a.dst.map(pos) - n_e)
    assert a[["i", "j"]].notna().all().all(), "hay aristas con extremos fuera de los vértices"
    return v, a.astype({"i": int, "j": int}), n_e

def pesos(a, peso):
    m = a["monto"].fillna(0).clip(lower=0)
    return {"binario": np.ones(len(a)), "n_adj": a["n_adj"].astype(float).to_numpy(),
            "monto": m.clip(lower=1).to_numpy(), "log_monto": (1 + np.log1p(m)).to_numpy()}[peso]

def matriz_bip(a, n_e, n_p, peso):
    return sp.csr_matrix((pesos(a, peso), (a.i, a.j)), shape=(n_e, n_p))

def matriz_grafo(A, dirigido):
    n_e, n_p = A.shape
    Z = lambda f, c: sp.csr_matrix((f, c))
    return sp.bmat([[Z(n_e, n_e), A], [Z(n_p, n_e) if dirigido else A.T, Z(n_p, n_p)]], format="csr")

### Algoritmos

- `pagerank_local`: Google matrix G = d·(P + a·vᵀ) + (1 − d)·1·vᵀ

- P estocástica por filas, a = nodos colgantes y v uniforme

- `hits_local`: a = Aᵀh, h = A·a, normalizando en L2; converge a los primeros vectores singulares de A

In [ ]:
def pagerank_local(M, d=DAMPING, tol=TOL, max_iter=MAX_ITER):
    n = M.shape[0]
    s = np.asarray(M.sum(axis=1)).ravel()
    colgante = s == 0
    PT = (sp.diags(np.divide(1.0, s, out=np.zeros(n), where=~colgante)) @ M).T.tocsr()
    v = np.full(n, 1.0 / n)
    r = v.copy()
    for k in range(1, max_iter + 1):
        nuevo = d * (PT @ r + r[colgante].sum() * v) + (1 - d) * v
        err = np.abs(nuevo - r).sum()
        r = nuevo
        if err < tol:
            break
    return r / r.sum(), k, err

def hits_local(A, tol=TOL, max_iter=MAX_ITER):
    AT = A.T.tocsr()
    h = np.full(A.shape[0], 1 / np.sqrt(A.shape[0]))
    for k in range(1, max_iter + 1):
        a = AT @ h
        a /= np.linalg.norm(a)
        nuevo = A @ a
        nuevo /= np.linalg.norm(nuevo)
        err = np.abs(nuevo - h).sum()
        h = nuevo
        if err < tol:
            break
    a = AT @ h
    return h, a / np.linalg.norm(a), k, err

### Métricas

- `spearman`, `top_jaccard`: coincidencia de dos rankings (completo y en el top-k)

- `gini`, `cuota_top`: concentración del puntaje

- `pct_por_tipo`: percentil dentro de entidades o dentro de proveedores

In [ ]:
def spearman(x, y):
    return float(spearmanr(np.asarray(x), np.asarray(y)).correlation)

def top_jaccard(x, y, k=TOP_K):
    a, b = set(np.argsort(-np.asarray(x))[:k]), set(np.argsort(-np.asarray(y))[:k])
    return len(a & b) / len(a | b)

def gini(x):
    x = np.sort(np.asarray(x, float)); n = len(x)
    return float((2 * np.arange(1, n + 1) - n - 1) @ x / (n * x.sum()))

def cuota_top(x, frac=0.01):
    x = np.sort(np.asarray(x, float))[::-1]
    return float(x[:max(1, int(len(x) * frac))].sum() / x.sum())

def pct_por_tipo(v, col):
    return v.groupby("tipo")[col].rank(pct=True)

### Clustering

`tipologia`: etiqueta KMeans de cada actor con el nombre de su tipología

In [ ]:
def tipologia(s, key):
    base = clus_dir
    ruta_tip = base / "e_analisis" / "tablas" / "tipologias"
    if not ruta_tip.exists():
        raise FileNotFoundError("Falta 02_Clustering/temp/e_analisis/tablas/tipologias: correr e_analisis")
    nombres = pd.read_parquet(ruta_tip)
    nombres = nombres.loc[nombres.sujeto == s, ["kmeans", "tipologia"]]
    for f in (f"kmeans_{s}", f"prep_{s}"):
        p = base / "datos" / f
        if not p.exists():
            continue
        d = pd.read_parquet(p)
        if "kmeans" not in d.columns and "cluster" in d.columns:
            d = d.rename(columns={"cluster": "kmeans"})      # b_kmeans guarda la etiqueta como "cluster"
        if "kmeans" in d.columns and key in d.columns:
            return d[[key, "kmeans"]].merge(nombres, on="kmeans", how="left")
    raise FileNotFoundError(f"Faltan las etiquetas KMeans de {s}: correr b_kmeans de 02_Clustering")

In [ ]:
def rellenar_vacias(t):
    t = t.copy()
    for c in t.columns:
        if t[c].dtype == object and t[c].isna().all():
            t[c] = "sin dato"
    return t


_guardar_tablas_base = guardar_tablas


def guardar_tablas(tablas, *args, **kwargs):
    vacias = [k for k, t in tablas.items() if len(t) == 0]
    if vacias:
        print(f"Aviso: tablas sin filas, no se escriben: {', '.join(vacias)}")
    return _guardar_tablas_base({k: rellenar_vacias(t) for k, t in tablas.items() if len(t) > 0}, *args, **kwargs)